In [8]:
import re
import pymupdf4llm
from llama_index.core import Document

def clean_medical_text(text):
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
    text = re.sub(r'^\s*\d+\s*$\n', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.replace('\x00', '')
    return text

pdf_path = r"C:\Users\dell\OneDrive\Desktop\c++ folder\Ai hacathon creativa\raggers\book800.pdf"
md_path = "book800.md"

# 1. Extract markdown from the PDF
raw_md_text = pymupdf4llm.to_markdown(pdf_path)

# 2. Clean it
cleaned_text = clean_medical_text(raw_md_text)

# 3. Save the markdown for later use (optional)
with open(md_path, "w", encoding="utf-8") as f:
    f.write(cleaned_text)

# 4. Create a LlamaIndex Document
docs = [Document(text=cleaned_text)]

In [9]:
import re
import pymupdf4llm
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex
import pandas as pd

# ---------- 1. Extract & Clean PDF ----------
def clean_medical_text(text):
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
    text = re.sub(r'^\s*\d+\s*$\n', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.replace('\x00', '')
    return text


# Save (optional)
with open("book800.md", "w", encoding="utf-8") as f:
    f.write(cleaned_text)



In [5]:
formatted_test_set = [
    {
        "query": "What are the main categories of genes involved in the development of cancer?",
        "expected_keywords": [
            "tumour suppressor genes",
            "proto-oncogenes",
            "DNA repair genes"
        ]
    },
    {
        "query": "What is the role of the MDT (multidisciplinary team) in cancer management?",
        "expected_keywords": [
            "diagnostic and staging procedures",
            "primary treatment approach",
            "adjuvant therapy",
            "rehabilitation",
            "palliative care"
        ]
    },
    {
        "query": "What is the most effective treatment for severe hypersomnia sleep-apnea syndrome (HSA) and what clinical improvements are observed?",
        "expected_keywords": [
            "open tracheal airway (tracheostomy)",
            "reduction in daytime sleepiness",
            "decrease in systemic hypertension",
            "fall in abnormally high haematocrit"
        ]
    },
    {
        "query": "What are the common indications for palliative surgery in cancer patients?",
        "expected_keywords": [
            "bowel obstruction",
            "fistulae",
            "jaundice",
            "ascites",
            "pain",
            "gastrointestinal bleeding"
        ]
    },
    {
        "query": "What is the principle behind combination chemotherapy?",
        "expected_keywords": [
            "each drug should have single-agent activity",
            "different mechanism of activity",
            "non-overlapping toxicity patterns",
            "different parts of the cell cycle",
            "not share same resistance mechanisms"
        ]
    },
    {
        "query": "What are the late effects of chemotherapy and radiotherapy on the heart?",
        "expected_keywords": [
            "anthracycline exposure",
            "dilated cardiomyopathy",
            "trastuzumab",
            "radiotherapy",
            "coronary artery disease"
        ]
    },
    {
        "query": "What is the role of hormone therapy in the treatment of breast cancer?",
        "expected_keywords": [
            "deplete circulating level of hormone",
            "block binding to receptors",
            "tumour regression",
            "apoptosis"
        ]
    },
    {
        "query": "What are the main types of targeted therapies used in oncology?",
        "expected_keywords": [
            "small molecule inhibitors",
            "monoclonal antibodies",
            "active immunotherapy",
            "adoptive immunotherapy",
            "tumour vaccines"
        ]
    },
    {
        "query": "What are the common oncological emergencies that require urgent management?",
        "expected_keywords": [
            "spinal cord compression",
            "bone marrow suppression",
            "superior vena cava obstruction",
            "raised intracranial pressure",
            "airway obstruction",
            "thromboembolic",
            "biochemical crises"
        ]
    },
    {
        "query": "What are the main principles of symptom control in palliative care?",
        "expected_keywords": [
            "pain management",
            "nausea and vomiting",
            "constipation",
            "cachexia and anorexia",
            "respiratory symptoms",
            "psychological distress"
        ]
    }
]

In [10]:

# Create LlamaIndex document
docs = [Document(text=cleaned_text)]

# ---------- 2. Chunking ----------
text_splitter = SentenceSplitter(chunk_size=800, chunk_overlap=150)
nodes = text_splitter.get_nodes_from_documents(docs)
print(f"Total chunks: {len(nodes)}")

Total chunks: 633


In [26]:
from typing import List
from llama_index.core.embeddings import BaseEmbedding
from sentence_transformers import SentenceTransformer

class ZEmbedEmbedding(BaseEmbedding):
    """Wrapper for zeroentropy/zembed-1 with separate query/document encoding."""

    def __init__(self, model_name: str = "zeroentropy/zembed-1", **kwargs):
        # Call parent __init__ without extra kwargs that are not fields
        super().__init__()
        # Store the model as a private attribute (not a Pydantic field)
        self._model = SentenceTransformer(
            model_name,
            trust_remote_code=True,
            model_kwargs={"torch_dtype": "float32"},  # Use float32 for compatibility
        )
        self._embedding_dim = 2560

    def _get_query_embedding(self, query: str) -> List[float]:
        emb = self._model.encode_query(query)
        return emb.tolist()

    def _get_text_embedding(self, text: str) -> List[float]:
        emb = self._model.encode_document([text])[0]
        return emb.tolist()

    def _get_text_embeddings(self, texts: List[str]) -> List[List[float]]:
        embs = self._model.encode_document(texts)
        return [emb.tolist() for emb in embs]

    # Async versions (required by BaseEmbedding)
    async def _aget_query_embedding(self, query: str) -> List[float]:
        return self._get_query_embedding(query)

    async def _aget_text_embedding(self, text: str) -> List[float]:
        return self._get_text_embedding(text)

    async def _aget_text_embeddings(self, texts: List[str]) -> List[List[float]]:
        return self._get_text_embeddings(texts)

    @property
    def class_name(self) -> str:
        return "ZEmbedEmbedding"

    @property
    def embedding_dim(self) -> int:
        return self._embedding_dim

In [27]:
# Build index with ZEmbed
embed_medical = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
embed_general = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

index_med = VectorStoreIndex(nodes, embed_model=embed_medical)
index_gen = VectorStoreIndex(nodes, embed_model=embed_general)
embed_zembed = ZEmbedEmbedding()
index_zembed = VectorStoreIndex(nodes, embed_model=embed_zembed)
print("ZEmbed index built successfully!")

Loading weights: 100%|██████████| 398/398 [00:28<00:00, 14.08it/s]


TypeError: 'str' object is not callable

In [21]:
# ---------- Evaluate all three models ----------
def evaluate_llamaindex_retriever(index, model_name, test_data, k=5):
    retriever = index.as_retriever(similarity_top_k=k)
    results = []
    for item in test_data:
        query = item["query"]
        expected = [kw.lower() for kw in item["expected_keywords"]]
        retrieved_nodes = retriever.retrieve(query)
        relevant_count = 0
        scores = []
        for node in retrieved_nodes:
            text = node.text.lower()
            scores.append(round(node.score, 4))
            if any(kw in text for kw in expected):
                relevant_count += 1
        precision_at_k = relevant_count / k
        avg_score = sum(scores) / len(scores) if scores else 0
        results.append({
            "model": model_name,
            "query": query,
            "precision@k": precision_at_k,
            "avg_similarity_score": avg_score,
        })
    return pd.DataFrame(results)

NameError: name 'index_zembed' is not defined

In [16]:
# Evaluate all three
df_med = evaluate_llamaindex_retriever(index_med, "PubMedBERT", formatted_test_set, k=5)
df_gen = evaluate_llamaindex_retriever(index_gen, "BGE-Large", formatted_test_set, k=5)
df_zembed = evaluate_llamaindex_retriever(index_zembed, "ZEmbed", formatted_test_set, k=5)

# Combine and summarize
comparison_df = pd.concat([df_med, df_gen, df_zembed], ignore_index=True)
summary = comparison_df.groupby("model").agg({
    "precision@k": "mean",
    "avg_similarity_score": "mean"
}).reset_index()

print("\n--- LLAMAINDEX EVALUATION SUMMARY ---")
print(summary)

NameError: name 'index_med' is not defined